# Data Preprocessing for Recommendation System

## Objective

In this notebook, we preprocess the MovieLens 100K dataset to prepare it for building recommendation models.

The preprocessing pipeline includes:

- Loading the raw dataset
- Data validation
- Creating the User–Movie Interaction Matrix
- Sparsity Analysis
- Label Encoding of User IDs and Movie IDs
- Train-Test Split
- Saving the processed dataset

This notebook generates the cleaned and encoded dataset that will be used in subsequent recommendation models.

In [ ]:
# Data Preprocessing for Recommendation System

## Objective

In this notebook, we preprocess the MovieLens 100K dataset to prepare it for building recommendation models.

The preprocessing pipeline includes:

- Loading the raw dataset
- Data validation
- Creating the User–Movie Interaction Matrix
- Sparsity Analysis
- Label Encoding of User IDs and Movie IDs
- Train-Test Split
- Saving the processed dataset

This notebook generates the cleaned and encoded dataset that will be used in subsequent recommendation models.

In [2]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

import warnings
warnings.filterwarnings("ignore")

print("Libraries Loaded Successfully")

Libraries Loaded Successfully


## Step 1: Dataset Validation

Before training any machine learning model, it is important to verify the quality of the dataset.

In this step, we check:

- Dataset shape
- Missing values
- Duplicate records
- Number of unique users
- Number of unique movies

This ensures the dataset is clean and ready for preprocessing.

In [4]:
ratings = pd.read_csv(
    "data/ml-100k/u.data",
    sep="\t",
    names=["user_id", "movie_id", "rating", "timestamp"]
)

movies = pd.read_csv(
    "data/ml-100k/u.item",
    sep="|",
    encoding="latin-1",
    header=None,
    usecols=[0, 1],
    names=["movie_id", "movie_title"]
)

movie_ratings = pd.merge(ratings, movies, on="movie_id")

movie_ratings.head()

,user_id,movie_id,rating,timestamp,movie_title
0,196,242,3,881250949,Kolya (1996)
1,186,302,3,891717742,L.A. Confidential (1997)
2,22,377,1,878887116,Heavyweights (1994)
3,244,51,2,880606923,Legends of the Fall (1994)
4,166,346,1,886397596,Jackie Brown (1997)


In [5]:
ratings = pd.read_csv(
    "data/ml-100k/u.data",
    sep="\t",
    names=["user_id", "movie_id", "rating", "timestamp"]
)

movies = pd.read_csv(
    "data/ml-100k/u.item",
    sep="|",
    encoding="latin-1",
    header=None,
    usecols=[0, 1],
    names=["movie_id", "movie_title"]
)

movie_ratings = pd.merge(ratings, movies, on="movie_id")

movie_ratings.head()

,user_id,movie_id,rating,timestamp,movie_title
0,196,242,3,881250949,Kolya (1996)
1,186,302,3,891717742,L.A. Confidential (1997)
2,22,377,1,878887116,Heavyweights (1994)
3,244,51,2,880606923,Legends of the Fall (1994)
4,166,346,1,886397596,Jackie Brown (1997)


In [6]:
import pandas as pd
import numpy as np

ratings = pd.read_csv(
    "data/ml-100k/u.data",
    sep="\t",
    names=["user_id", "movie_id", "rating", "timestamp"]
)

movies = pd.read_csv(
    "data/ml-100k/u.item",
    sep="|",
    encoding="latin-1",
    header=None,
    usecols=[0, 1],
    names=["movie_id", "movie_title"]
)

movie_ratings = pd.merge(
    ratings,
    movies,
    on="movie_id"
)

print("✅ Dataset Loaded Successfully")
movie_ratings.head()

✅ Dataset Loaded Successfully


,user_id,movie_id,rating,timestamp,movie_title
0,196,242,3,881250949,Kolya (1996)
1,186,302,3,891717742,L.A. Confidential (1997)
2,22,377,1,878887116,Heavyweights (1994)
3,244,51,2,880606923,Legends of the Fall (1994)
4,166,346,1,886397596,Jackie Brown (1997)


In [7]:
print("="*50)
print("DATA VALIDATION")
print("="*50)

print("Shape :", movie_ratings.shape)

print("\nMissing Values")
print(movie_ratings.isnull().sum())

print("\nDuplicate Rows")
print(movie_ratings.duplicated().sum())

print("\nUnique Users :", movie_ratings["user_id"].nunique())
print("Unique Movies :", movie_ratings["movie_id"].nunique())

DATA VALIDATION
Shape : (100000, 5)

Missing Values
user_id        0
movie_id       0
rating         0
timestamp      0
movie_title    0
dtype: int64

Duplicate Rows
0

Unique Users : 943
Unique Movies : 1682


## Step 2: User–Movie Interaction Matrix

Recommendation systems rely on user-item interactions.

The interaction matrix represents:

- Rows → Users
- Columns → Movies
- Values → Ratings given by users

Missing values (NaN) indicate that a user has not rated a particular movie.

This matrix is widely used in collaborative filtering techniques.

In [8]:
interaction_matrix = movie_ratings.pivot_table(
    index="user_id",
    columns="movie_id",
    values="rating"
)

interaction_matrix.head()

movie_id,1,2,3,4,5,6,7,8,9,10,...,1673,1674,1675,1676,1677,1678,1679,1680,1681,1682
user_id,,,,,,,,,,,,,,,,,,,,,
1,5.0,3.0,4.0,3.0,3.0,5.0,4.0,1.0,5.0,3.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,4.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
print("Interaction Matrix Shape:")
print(interaction_matrix.shape)

Interaction Matrix Shape:
(943, 1682)


## Step 3: Sparsity Analysis

Real-world recommendation datasets are highly sparse because each user interacts with only a small fraction of the available items.

Dataset sparsity is calculated as:

Sparsity (%) = (1 − Number of Filled Cells / Total Cells) × 100

Understanding sparsity helps explain why recommendation systems are challenging and motivates the use of advanced algorithms.

In [10]:
total_cells = interaction_matrix.shape[0] * interaction_matrix.shape[1]

filled_cells = interaction_matrix.count().sum()

sparsity = 100 * (1 - filled_cells / total_cells)

print(f"Total Cells       : {total_cells:,}")
print(f"Filled Cells      : {filled_cells:,}")
print(f"Sparsity (%)      : {sparsity:.2f}%")

Total Cells       : 1,586,126
Filled Cells      : 100,000
Sparsity (%)      : 93.70%


In [11]:
from sklearn.preprocessing import LabelEncoder

# Create encoders
user_encoder = LabelEncoder()
movie_encoder = LabelEncoder()

# Encode IDs
movie_ratings["user_encoded"] = user_encoder.fit_transform(movie_ratings["user_id"])

movie_ratings["movie_encoded"] = movie_encoder.fit_transform(movie_ratings["movie_id"])

# Show first few rows
movie_ratings.head()

,user_id,movie_id,rating,timestamp,movie_title,user_encoded,movie_encoded
0,196,242,3,881250949,Kolya (1996),195,241
1,186,302,3,891717742,L.A. Confidential (1997),185,301
2,22,377,1,878887116,Heavyweights (1994),21,376
3,244,51,2,880606923,Legends of the Fall (1994),243,50
4,166,346,1,886397596,Jackie Brown (1997),165,345


## Step 4: Label Encoding

Deep learning models require user IDs and movie IDs to be represented as continuous integer indices starting from 0.

Label Encoding converts the original IDs into encoded values that can be directly used by embedding layers in PyTorch.

In [12]:
print("Original User IDs:")
print(movie_ratings["user_id"].head().tolist())

print()

print("Encoded User IDs:")
print(movie_ratings["user_encoded"].head().tolist())

print()

print("Original Movie IDs:")
print(movie_ratings["movie_id"].head().tolist())

print()

print("Encoded Movie IDs:")
print(movie_ratings["movie_encoded"].head().tolist())

Original User IDs:
[196, 186, 22, 244, 166]

Encoded User IDs:
[195, 185, 21, 243, 165]

Original Movie IDs:
[242, 302, 377, 51, 346]

Encoded Movie IDs:
[241, 301, 376, 50, 345]


In [13]:
print("Number of Encoded Users :", movie_ratings["user_encoded"].nunique())

print("Number of Encoded Movies :", movie_ratings["movie_encoded"].nunique())

Number of Encoded Users : 943
Number of Encoded Movies : 1682


## Step 5: Train-Test Split

To evaluate recommendation models fairly, the dataset is divided into:

- Training Set (80%)
- Testing Set (20%)

The model learns patterns from the training data and is evaluated on previously unseen interactions in the testing data.

In [14]:
from sklearn.model_selection import train_test_split

train_data, test_data = train_test_split(
    movie_ratings,
    test_size=0.2,
    random_state=42
)

print("Training Samples :", len(train_data))
print("Testing Samples  :", len(test_data))

Training Samples : 80000
Testing Samples  : 20000


In [15]:
print(train_data.head())

print()

print(test_data.head())

       user_id  movie_id  rating  timestamp                  movie_title  \
75220      807      1411       1  893082619            Barbarella (1968)   
48955      474       659       5  887925187  Arsenic and Old Lace (1944)   
44966      463       268       4  877384940           Chasing Amy (1997)   
13568      139       286       4  879537844  English Patient, The (1996)   
92727      621       751       4  883799651   Tomorrow Never Dies (1997)   

       user_encoded  movie_encoded  
75220           806           1410  
48955           473            658  
44966           462            267  
13568           138            285  
92727           620            750  

       user_id  movie_id  rating  timestamp                   movie_title  \
75721      877       381       4  882677345       Muriel's Wedding (1994)   
80184      815       602       3  878694269  American in Paris, An (1951)   
19864       94       431       4  891721716             Highlander (1986)   
76699      4

In [16]:
print("Training Users :", train_data["user_encoded"].nunique())
print("Training Movies:", train_data["movie_encoded"].nunique())

print()

print("Testing Users :", test_data["user_encoded"].nunique())
print("Testing Movies:", test_data["movie_encoded"].nunique())

Training Users : 943
Training Movies: 1653

Testing Users : 940
Testing Movies: 1411


## Step 6: Save Processed Dataset

The processed dataset is saved for future notebooks.

This avoids repeating preprocessing steps and follows the standard machine learning workflow:

Raw Data → Preprocessing → Model Training → Evaluation

In [17]:


import os

os.makedirs("processed_data", exist_ok=True)

movie_ratings.to_csv(
    "processed_data/movie_ratings_processed.csv",
    index=False
)

print("✅ Processed dataset saved successfully!")

✅ Processed dataset saved successfully!


# Conclusion

In this notebook, we successfully prepared the MovieLens 100K dataset for recommendation modeling.

The following preprocessing tasks were completed:

- Dataset validation
- User–Movie interaction matrix creation
- Sparsity analysis
- Label encoding
- Train-test split
- Processed dataset saved

The processed dataset is now ready for building recommendation algorithms in the next notebook.

In [18]:
import os

print(os.listdir("data"))

['ml-100k']


In [19]:
import pandas as pd

# Ratings
ratings = pd.read_csv(
    "data/ml-100k/u.data",
    sep="\t",
    names=["user_id", "movie_id", "rating", "timestamp"]
)

# Movies
movies = pd.read_csv(
    "data/ml-100k/u.item",
    sep="|",
    encoding="latin-1",
    usecols=[0, 1],
    names=["movie_id", "movie_title"]
)

# Merge
movie_ratings = pd.merge(
    ratings,
    movies,
    on="movie_id"
)

print("Files Loaded Successfully")
movie_ratings.head()

Files Loaded Successfully


,user_id,movie_id,rating,timestamp,movie_title
0,196,242,3,881250949,Kolya (1996)
1,186,302,3,891717742,L.A. Confidential (1997)
2,22,377,1,878887116,Heavyweights (1994)
3,244,51,2,880606923,Legends of the Fall (1994)
4,166,346,1,886397596,Jackie Brown (1997)


In [20]:
import faiss
import pandas as pd

index = faiss.read_index("models/user_faiss.index")

user_embeddings = pd.read_csv("models/user_embeddings.csv")

user_mapping = pd.read_csv("models/user_mapping.csv")

print("Everything Loaded Successfully!")

Everything Loaded Successfully!
